In [ ]:
combo = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID']

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window


def get_non_unique_rows(df, combo, limit_display=20):
    """Return rows whose values are duplicated for the supplied key columns."""
    print(f"🔍 Recherche des doublons pour la combinaison : {combo}...\n")

    window_spec = Window.partitionBy(*[F.col(c) for c in combo])
    duplicates_df = (
        df.withColumn("_count_occurrence", F.count("*").over(window_spec))
        .filter(F.col("_count_occurrence") > 1)
        .drop("_count_occurrence")
    )

    total_duplicate_rows = duplicates_df.count()
    if total_duplicate_rows == 0:
        print(f"✅ La combinaison {combo} est unique.")
        return df.sparkSession.createDataFrame([], df.schema)

    distinct_duplicate_keys = duplicates_df.select(*combo).distinct().count()
    print(f"⚠️ {total_duplicate_rows:,} lignes impliquées dans {distinct_duplicate_keys:,} clés dupliquées.")
    duplicates_df.orderBy(*combo).show(limit_display, truncate=False)
    return duplicates_df

In [ ]:
duplicates_df = get_non_unique_rows(
    spark.sql("SELECT * FROM nyc_taxi.bronze.green_taxi"),
    combo,
)

In [ ]:
from datetime import datetime

quarantine_schema = "nyc_taxi.quarantine"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {quarantine_schema}")

duplicates_df = duplicates_df.withColumn(
    "trip_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("lpep_pickup_datetime"),
            F.col("lpep_dropoff_datetime"),
            F.col("PULocationID"),
            F.col("DOLocationID"),
        ),
        256,
    ),
)


duplicates_df.write.mode("overwrite").saveAsTable(
    f"{quarantine_schema}.taxi_trips_duplicates_latest"
)